## A.SETUP

In [1]:
# Cell A1 - imports and data
import sys, pickle, time
sys.path.append("..")

import numpy as np
import networkx as nx
from src.cascade import to_igraph, loads, add_datacenter, cascade

with open("../data/processed/nordic_graph.pkl", "rb") as f:
    G = pickle.load(f)

g = to_igraph(G)
L0 = loads(g)                       # baseline loads, compute once and reuse
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

1763 nodes, 2253 edges


## B.Checks

In [2]:
# Cell B1 - graph and speed checks, igraph vs NetworkX
print("components:", nx.number_connected_components(G))
costs = [d["cost"] for _, _, d in G.edges(data=True)]
print("cost type:", type(costs[0]).__name__, "| example:", costs[:3])

t = time.time()
bc = nx.betweenness_centrality(G, weight="cost", normalized=False)
print(f"NetworkX betweenness: {time.time() - t:.1f} s")

t = time.time()
bc_ig = dict(zip(g.vs["name"], g.betweenness(weights="cost")))
print(f"igraph betweenness:   {time.time() - t:.2f} s")

print("largest difference:", max(abs(bc[n] - bc_ig[n]) for n in G))

components: 1
cost type: int | example: [2034, 2034, 147]
NetworkX betweenness: 15.3 s
igraph betweenness:   0.41 s
largest difference: 1.4551915228366852e-11


## C.Experiments

In [4]:
# Cell C1 - first test: baseline cascade and datacenter
top = max(L0, key=L0.get)           # most loaded substation

t = time.time()
failed, lcc = cascade(g, alpha=0.2, triggers=[top], L0=L0)
print(f"baseline: {len(failed)} failed, LCC {lcc:.2%}, {time.time() - t:.1f} s")

h = add_datacenter(g, top)
failed, lcc = cascade(g, alpha=0.2, h=h, dc="DC", demand=1.0, L0=L0)
print(f"datacenter: {len(failed)} failed, LCC {lcc:.2%}")

baseline: 580 failed, LCC 9.47%, 0.6 s
datacenter: 0 failed, LCC 100.00%


In [6]:
# Cell C2 - at what demand does the first substation overload?
def demand_threshold(g, bus, alpha, L0):
    h = add_datacenter(g, bus)
    d = h.vs.find(name="DC").index
    others = [v.index for v in h.vs if v.index != d]
    extra = dict(zip(h.vs["name"],
                     h.betweenness(weights="cost", sources=others, targets=[d])))
    th = {n: alpha * L0[n] / extra[n] for n in L0 if extra.get(n, 0) > 0}
    first = min(th, key=th.get)
    return th[first], first

value, node = demand_threshold(g, top, alpha=0.2, L0=L0)
print(f"first overload at demand ≈ {value:.0f} (node {node})")

first overload at demand ≈ 100 (node way/29309786-380)


In [7]:
#C3
for demand in [50, 100, 150, 200, 300]:
    h = add_datacenter(g, top)
    failed, lcc = cascade(g, alpha=0.2, h=h, dc="DC", demand=demand, L0=L0)
    print(f"demand {demand:>3}: {len(failed):>4} failed, LCC {lcc:.2%}")

demand  50:    0 failed, LCC 100.00%
demand 100:  580 failed, LCC 9.47%
demand 150:  524 failed, LCC 19.57%
demand 200:  533 failed, LCC 19.63%
demand 300:  442 failed, LCC 26.32%


In [8]:
#C4 
hv = [n for n, d in G.nodes(data=True) if d.get("voltage", 0) >= 380]
candidates = sorted(hv, key=L0.get, reverse=True)[:20]   # placeholder list

rows = []
for c in candidates:
    value, node = demand_threshold(g, c, alpha=0.2, L0=L0)
    rows.append((c, G.nodes[c]["country"], round(value), node))

for c, country, value, node in sorted(rows, key=lambda r: -r[2]):
    print(f"{country}  {c:<28} threshold {value:>6}  (first to fail: {node})")

DE  way/29309786-380             threshold    100  (first to fail: way/29309786-380)
DE  way/525964629-380            threshold     94  (first to fail: way/525964629-380)
NO  relation/8890887-420         threshold     94  (first to fail: relation/8890887-420)
DE  way/525964617                threshold     93  (first to fail: way/525964617)
NO  relation/8890887             threshold     93  (first to fail: relation/8890887)
DE  DE10-380                     threshold     79  (first to fail: way/11285720-380)
DE  way/157328339-380            threshold     74  (first to fail: way/157328339-380)
DE  DE6-380                      threshold     71  (first to fail: DE6-380)
DE  way/1107961833-380           threshold     71  (first to fail: way/1107961833-380)
DE  DE11-380                     threshold     68  (first to fail: way/11285720-380)
DE  DE8-380                      threshold     65  (first to fail: way/25306938-380)
DE  way/456386244-380            threshold     64  (first to fail: wa

In [9]:
#C5
results = []
for c in candidates:
    for demand in [30, 60, 100]:
        h = add_datacenter(g, c)
        failed, lcc = cascade(g, alpha=0.2, h=h, dc="DC", demand=demand, L0=L0)
        results.append({"candidate": c, "country": G.nodes[c]["country"],
                        "demand": demand, "failed": len(failed), "lcc": lcc})

import pandas as pd
df = pd.DataFrame(results)
print(df.pivot(index="candidate", columns="demand", values="failed"))
df.to_csv("../results/datacenter_immediate.csv", index=False)

demand                 30   60   100
candidate                           
DE10-380                 0    0  317
DE11-380                 0    0  322
DE6-380                  0    0  367
DE8-380                  0    0  405
SE35-400                 0  418  312
SE35-420                 0  419  312
relation/10153505-400    0  384  312
relation/8890887         0    0  450
relation/8890887-420     0    0  428
way/1107961833-380       0    0  404
way/157328339-380        0    0  485
way/187832596-400      287  330  344
way/24413992-380         0    3  478
way/27739871-380         0    3  388
way/29309786-380         0    0  580
way/456386244-380        0    0  476
way/525964617            0    0  424
way/525964629-380        0    0  411
way/60495669-420         0  409  314
way/955268863-380        0    3  372


In [10]:
# Cell C6 - baseline cascades: no datacenter, single failures
import random
import pandas as pd

top10 = sorted(L0, key=L0.get, reverse=True)[:10]
rand10 = random.Random(42).sample([n for n in L0 if n not in top10], 10)

rows = []
for alpha in [0.2, 0.5]:
    for kind, triggers in [("top-10", top10), ("random", rand10)]:
        for t in triggers:
            failed, lcc = cascade(g, alpha=alpha, triggers=[t], L0=L0)
            rows.append({"alpha": alpha, "trigger_type": kind, "trigger": t,
                         "failed": len(failed), "lcc": lcc})

base = pd.DataFrame(rows)
base.to_csv("../results/baseline_cascades.csv", index=False)
print(base.groupby(["alpha", "trigger_type"])[["failed", "lcc"]].mean().round(3))

                    failed    lcc
alpha trigger_type               
0.2   random         120.0  0.795
      top-10         443.9  0.223
0.5   random          79.9  0.872
      top-10         347.5  0.289
